# 01 Data Ingestion: SEC Filings

## Goal

This notebook downloads real SEC filing metadata and raw 10-K filings for the RiskRadar AI project.

We will use the SEC submissions API to collect filing metadata for public companies, then identify and download each company's latest 10-K filing.

The output from this notebook will become the raw document layer for the RAG system.

In [1]:
from pathlib import Path
import time
import json
import re

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

SEC_RAW_DIR = RAW_DIR / "sec_filings"
SEC_METADATA_DIR = RAW_DIR / "sec_metadata"

SEC_RAW_DIR.mkdir(parents=True, exist_ok=True)
SEC_METADATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("SEC raw filing folder:", SEC_RAW_DIR)
print("SEC metadata folder:", SEC_METADATA_DIR)

Project root: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI
SEC raw filing folder: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\raw\sec_filings
SEC metadata folder: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\raw\sec_metadata


In [3]:
SEC_HEADERS = {
    "User-Agent": "RiskRadarAI/0.1 tevinswright@gmail.com",
    "Accept-Encoding": "gzip, deflate",
    "Host": "data.sec.gov"
}

SEC_ARCHIVE_HEADERS = {
    "User-Agent": "RiskRadarAI/0.1 tevinswright@gmail.com",
    "Accept-Encoding": "gzip, deflate",
    "Host": "www.sec.gov"
}

SEC may block undeclared automated tools.
A clear User-Agent tells the SEC who is making the request.

#### Load Company Universe

In [4]:
# Set the path to the company universe file from notebook 00
company_file = PROCESSED_DIR / "company_universe_with_cik.csv"

# Check that the file exists before loading it
if not company_file.exists():
    raise FileNotFoundError(
        f"Could not find {company_file}. Run 00_project_intro.ipynb first."
    )

# Load the company universe with ticker and CIK information
company_universe = pd.read_csv(company_file)

In [5]:
# Convert CIK into a 10-digit string required by the SEC submissions API
company_universe["cik_padded"] = (
    company_universe["cik"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(10)
)

company_universe.head()


,ticker,company_name,sector,cik,cik_padded,company_title
0,NVDA,NVIDIA,Semiconductors,1045810,0001045810,NVIDIA CORP
1,MSFT,Microsoft,Cloud / Software,789019,0000789019,MICROSOFT CORP
2,AAPL,Apple,Consumer Technology,320193,0000320193,Apple Inc.
3,TSLA,Tesla,Electric Vehicles,1318605,0001318605,"Tesla, Inc."
4,AMZN,Amazon,E-Commerce / Cloud,1018724,0001018724,AMAZON COM INC


In [7]:
# Choose a smaller group of companies for the first ingestion run
# This keeps the project fast while we build the pipeline
starter_tickers = ["NVDA", "MSFT", "AAPL", "TSLA", "AMD"]

# Filter the company universe to only our starter companies
starter_companies = company_universe[company_universe["ticker"].isin(starter_tickers)].copy()

# Sort by ticker so the output is easy to read
starter_companies = starter_companies.sort_values("ticker").reset_index(drop=True)

# Display the selected companies
starter_companies

,ticker,company_name,sector,cik,cik_padded,company_title
0,AAPL,Apple,Consumer Technology,320193,0000320193,Apple Inc.
1,AMD,Advanced Micro Devices,Semiconductors,2488,0000002488,ADVANCED MICRO DEVICES INC
2,MSFT,Microsoft,Cloud / Software,789019,0000789019,MICROSOFT CORP
3,NVDA,NVIDIA,Semiconductors,1045810,0001045810,NVIDIA CORP
4,TSLA,Tesla,Electric Vehicles,1318605,0001318605,"Tesla, Inc."


#### SEC request json function

In [8]:
def get_sec_json(url, headers=SEC_HEADERS, sleep_seconds=0.2):
    """
    Download JSON data from the SEC with basic error handling.
    """
    #pause briefly so we do not hit the SEC too aggressively
    time.sleep(sleep_seconds)

    # send Get request to the SEC endpoint
    response = requests.get(url, headers=headers)

    # raise a clear error if the request fail
    if response.status_code !=200:
        raise Exception(
            f"SEC request failed. Status code: {response.status_code}. URL: {url}"
        )
    
    # return the response as JSON
    return response.json()

#### Test SEC Submission Request

In [ ]:
# select the first starter company as a test case
sample_company = starter_companies.iloc[0]

# pull ticker and padded CIK from the selected comapny
sample_ticker = sample_company['ticker']
sample_cik = sample_company["cik_padded"]

# Build the sec submissions API URL
sample_url = f"https://data.sec.gov/submissions/CIK{sample_cik}.json"

# print the request information
print("Ticker:", sample_ticker)
print("CIK:", sample_cik)
print("URL:", sample_url)

Ticker: AAPL
CIK: 0000320193
URL: https://data.sec.gov/submissions/CIK0000320193.json


In [13]:
# download the sample company submission JSOn
sample_submission = get_sec_json(sample_url)

# show the top-level keys in the JSON response
sample_submission.keys()

dict_keys(['cik', 'entityType', 'sic', 'sicDescription', 'ownerOrg', 'insiderTransactionForOwnerExists', 'insiderTransactionForIssuerExists', 'name', 'tickers', 'exchanges', 'ein', 'lei', 'description', 'website', 'investorWebsite', 'category', 'fiscalYearEnd', 'stateOfIncorporation', 'stateOfIncorporationDescription', 'addresses', 'phone', 'flags', 'formerNames', 'filings'])

#### Inspect Sample Submission

In [14]:
# Print basic company information from the SEC response
print("Company name:", sample_submission["name"])
print("CIK:", sample_submission["cik"])

# Show the filing groups available in the response
print("Available filing groups:", sample_submission["filings"].keys())

Company name: Apple Inc.
CIK: 0000320193
Available filing groups: dict_keys(['recent', 'files'])


In [15]:
def submissions_to_dataframe(submission_json, ticker, company_name):
    """
    Convert SEC recent filing metadata into a pandas DataFrame.
    """

    # Extract the recent filing metadata from the SEC JSON
    recent_filings = submission_json["filings"]["recent"]

    # Convert the dictionary of lists into a DataFrame
    filings_df = pd.DataFrame(recent_filings)

    # Add ticker so we can identify the company later
    filings_df["ticker"] = ticker

    # Add company name from our project company universe
    filings_df["company_name"] = company_name

    # Add CIK from the SEC response
    filings_df["cik"] = submission_json["cik"]

    # Return the clean filing metadata table
    return filings_df

In [16]:
# Convert the sample SEC submission into a DataFrame
sample_filings = submissions_to_dataframe(
    submission_json=sample_submission,
    ticker=sample_ticker,
    company_name=sample_company["company_name"]
)

# Preview the first few filings
sample_filings.head()

,accessionNumber,filingDate,reportDate,acceptanceDateTime,act,form,fileNumber,filmNumber,items,core_type,size,isXBRL,isInlineXBRL,isXBRLNumeric,primaryDocument,primaryDocDescription,ticker,company_name,cik
0,0001140361-26-025622,2026-06-17,2026-06-15,2026-06-17T22:40:43.000Z,,4,,,,4,9220,0,0,0.0,xslF345X06/form4.xml,FORM 4,AAPL,Apple,0000320193
1,0001140361-26-025620,2026-06-17,2026-06-15,2026-06-17T22:30:15.000Z,,4,,,,4,10141,0,0,0.0,xslF345X06/form4.xml,FORM 4,AAPL,Apple,0000320193
2,0001140361-26-023363,2026-05-29,2026-05-27,2026-05-29T22:30:27.000Z,,4,,,,4,6956,0,0,0.0,xslF345X06/form4.xml,FORM 4,AAPL,Apple,0000320193
3,0001140361-26-023149,2026-05-28,,2026-05-28T20:30:18.000Z,34,SD,001-36743,261036676,,SD,31060,0,0,0.0,ef20073373_sd.htm,SD,AAPL,Apple,0000320193
4,0001921094-26-000555,2026-05-27,,2026-05-27T20:00:33.000Z,33,144,001-36743,261025990,,144,4960,0,0,0.0,xsl144X01/primary_doc.xml,,AAPL,Apple,0000320193


In [17]:
# Count filing types so we can see what forms are available
filing_type_counts = sample_filings["form"].value_counts()

# Show the most common filing types
filing_type_counts.head(15)

form
4           589
8-K         104
424B2        48
144          44
10-Q         33
PX14A6G      27
FWP          24
SC 13G/A     22
DEFA14A      12
SD           11
DEF 14A      11
10-K         11
3            11
NO ACT        7
8-A12B        5
Name: count, dtype: int64

In [18]:
# Filter only annual 10-K filings
sample_10k = sample_filings[sample_filings["form"] == "10-K"].copy()

# Sort by filing date so the newest filing is first
sample_10k = sample_10k.sort_values("filingDate", ascending=False)

# Display important metadata columns
sample_10k[
    [
        "ticker",
        "company_name",
        "form",
        "filingDate",
        "accessionNumber",
        "primaryDocument",
        "primaryDocDescription"
    ]
].head()

,ticker,company_name,form,filingDate,accessionNumber,primaryDocument,primaryDocDescription
51,AAPL,Apple,10-K,2025-10-31,0000320193-25-000079,aapl-20250927.htm,10-K
136,AAPL,Apple,10-K,2024-11-01,0000320193-24-000123,aapl-20240928.htm,10-K
230,AAPL,Apple,10-K,2023-11-03,0000320193-23-000106,aapl-20230930.htm,10-K
308,AAPL,Apple,10-K,2022-10-28,0000320193-22-000108,aapl-20220924.htm,10-K
383,AAPL,Apple,10-K,2021-10-29,0000320193-21-000105,aapl-20210925.htm,10-K


In [19]:
def build_filing_url(cik, accession_number, primary_document):
    """
    Build the SEC archive URL for a filing document.
    """

    # Remove leading zeros from CIK for the SEC archive URL
    cik_clean = str(int(cik))

    # Remove dashes from accession number for the SEC archive URL
    accession_clean = accession_number.replace("-", "")

    # Build the complete filing document URL
    filing_url = (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik_clean}/{accession_clean}/{primary_document}"
    )

    # Return the final URL
    return filing_url

In [20]:
# Select the latest sample 10-K row
sample_row = sample_10k.iloc[0]

# Build a filing URL for the selected 10-K
test_filing_url = build_filing_url(
    cik=sample_row["cik"],
    accession_number=sample_row["accessionNumber"],
    primary_document=sample_row["primaryDocument"]
)

# Display the filing URL
test_filing_url

'https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm'

#### Collect meta data

In [21]:
# Create an empty list to store each company's filing metadata
all_filings = []

# Loop through each starter company
for _, row in tqdm(starter_companies.iterrows(), total=len(starter_companies)):

    # Get company fields from the current row
    ticker = row["ticker"]
    company_name = row["company_name"]
    cik_padded = row["cik_padded"]

    # Build SEC submissions API URL
    url = f"https://data.sec.gov/submissions/CIK{cik_padded}.json"

    # Download submission JSON from the SEC
    submission_json = get_sec_json(url)

    # Convert SEC JSON into a DataFrame
    filings_df = submissions_to_dataframe(
        submission_json=submission_json,
        ticker=ticker,
        company_name=company_name
    )

    # Add this company's filing metadata to the list
    all_filings.append(filings_df)

# Combine all company filing tables into one DataFrame
all_filings_df = pd.concat(all_filings, ignore_index=True)

# Show the number of rows and columns
all_filings_df.shape

100%|██████████| 5/5 [00:02<00:00,  2.25it/s]


(5003, 19)

In [22]:
all_filings_df.head()

,accessionNumber,filingDate,reportDate,acceptanceDateTime,act,form,fileNumber,filmNumber,items,core_type,size,isXBRL,isInlineXBRL,isXBRLNumeric,primaryDocument,primaryDocDescription,ticker,company_name,cik
0,0001140361-26-025622,2026-06-17,2026-06-15,2026-06-17T22:40:43.000Z,,4,,,,4,9220,0,0,0.0,xslF345X06/form4.xml,FORM 4,AAPL,Apple,0000320193
1,0001140361-26-025620,2026-06-17,2026-06-15,2026-06-17T22:30:15.000Z,,4,,,,4,10141,0,0,0.0,xslF345X06/form4.xml,FORM 4,AAPL,Apple,0000320193
2,0001140361-26-023363,2026-05-29,2026-05-27,2026-05-29T22:30:27.000Z,,4,,,,4,6956,0,0,0.0,xslF345X06/form4.xml,FORM 4,AAPL,Apple,0000320193
3,0001140361-26-023149,2026-05-28,,2026-05-28T20:30:18.000Z,34,SD,001-36743,261036676,,SD,31060,0,0,0.0,ef20073373_sd.htm,SD,AAPL,Apple,0000320193
4,0001921094-26-000555,2026-05-27,,2026-05-27T20:00:33.000Z,33,144,001-36743,261025990,,144,4960,0,0,0.0,xsl144X01/primary_doc.xml,,AAPL,Apple,0000320193


In [23]:
# Display the key filing metadata columns
all_filings_df[
    [
        "ticker",
        "company_name",
        "form",
        "filingDate",
        "accessionNumber",
        "primaryDocument",
        "primaryDocDescription"
    ]
].head(20)

,ticker,company_name,form,filingDate,accessionNumber,primaryDocument,primaryDocDescription
0,AAPL,Apple,4,2026-06-17,0001140361-26-025622,xslF345X06/form4.xml,FORM 4
1,AAPL,Apple,4,2026-06-17,0001140361-26-025620,xslF345X06/form4.xml,FORM 4
2,AAPL,Apple,4,2026-05-29,0001140361-26-023363,xslF345X06/form4.xml,FORM 4
3,AAPL,Apple,SD,2026-05-28,0001140361-26-023149,ef20073373_sd.htm,SD
4,AAPL,Apple,144,2026-05-27,0001921094-26-000555,xsl144X01/primary_doc.xml,
5,AAPL,Apple,4,2026-05-12,0001140361-26-020871,xslF345X06/form4.xml,FORM 4
6,AAPL,Apple,4,2026-05-08,0001140361-26-020298,xslF345X06/form4.xml,FORM 4
7,AAPL,Apple,144,2026-05-06,0001921094-26-000446,xsl144X01/primary_doc.xml,
8,AAPL,Apple,144,2026-05-05,0001950047-26-004044,xsl144X01/primary_doc.xml,
9,AAPL,Apple,10-Q,2026-05-01,0000320193-26-000013,aapl-20260328.htm,10-Q


In [24]:
# Set output path for all recent filing metadata
all_filings_file = SEC_METADATA_DIR / "starter_company_all_filings.csv"

# Save all filing metadata to CSV
all_filings_df.to_csv(all_filings_file, index=False)

# Confirm where the file was saved
print("Saved all filing metadata to:", all_filings_file)

Saved all filing metadata to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\raw\sec_metadata\starter_company_all_filings.csv


In [25]:
# Filter only 10-K annual filings
ten_k_filings = all_filings_df[all_filings_df["form"] == "10-K"].copy()

# Sort by ticker and filing date so the latest 10-K appears first per company
ten_k_filings = ten_k_filings.sort_values(
    ["ticker", "filingDate"],
    ascending=[True, False]
)

# Keep the latest 10-K for each company
latest_10k = ten_k_filings.groupby("ticker").head(1).copy()

# Build filing URLs for each latest 10-K
latest_10k["filing_url"] = latest_10k.apply(
    lambda row: build_filing_url(
        cik=row["cik"],
        accession_number=row["accessionNumber"],
        primary_document=row["primaryDocument"]
    ),
    axis=1
)

# Display final latest 10-K metadata
latest_10k[
    [
        "ticker",
        "company_name",
        "form",
        "filingDate",
        "accessionNumber",
        "primaryDocument",
        "filing_url"
    ]
]

,ticker,company_name,form,filingDate,accessionNumber,primaryDocument,filing_url
51,AAPL,Apple,10-K,2025-10-31,0000320193-25-000079,aapl-20250927.htm,https://www.sec.gov/Archives/edgar/data/320193...
1070,AMD,Advanced Micro Devices,10-K,2026-02-04,0000002488-26-000018,amd-20251227.htm,https://www.sec.gov/Archives/edgar/data/2488/0...
2176,MSFT,Microsoft,10-K,2025-07-30,0000950170-25-100235,msft-20250630.htm,https://www.sec.gov/Archives/edgar/data/789019...
3055,NVDA,NVIDIA,10-K,2026-02-25,0001045810-26-000021,nvda-20260125.htm,https://www.sec.gov/Archives/edgar/data/104581...
4027,TSLA,Tesla,10-K,2026-01-29,0001628280-26-003952,tsla-20251231.htm,https://www.sec.gov/Archives/edgar/data/131860...


In [26]:
# Set output path for latest 10-K metadata
latest_10k_file = SEC_METADATA_DIR / "starter_company_latest_10k.csv"

# Save latest 10-K metadata to CSV
latest_10k.to_csv(latest_10k_file, index=False)

# Confirm where the file was saved
print("Saved latest 10-K metadata to:", latest_10k_file)

Saved latest 10-K metadata to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\raw\sec_metadata\starter_company_latest_10k.csv


In [27]:
def download_filing_text(filing_url, headers=SEC_ARCHIVE_HEADERS, sleep_seconds=0.2):
    """
    Download raw SEC filing document text or HTML.
    """

    # Pause briefly before making the request
    time.sleep(sleep_seconds)

    # Download the filing from the SEC archive
    response = requests.get(filing_url, headers=headers)

    # Raise a clear error if the download fails
    if response.status_code != 200:
        raise Exception(
            f"Filing download failed. Status code: {response.status_code}. URL: {filing_url}"
        )

    # Return the filing HTML/text
    return response.text

In [28]:
# Create an empty list to track downloaded files
downloaded_files = []

# Loop through each latest 10-K filing
for _, row in tqdm(latest_10k.iterrows(), total=len(latest_10k)):

    # Extract metadata from current filing row
    ticker = row["ticker"]
    company_name = row["company_name"]
    filing_date = row["filingDate"]
    accession_number = row["accessionNumber"]
    filing_url = row["filing_url"]

    # Download the raw filing HTML/text
    raw_text = download_filing_text(filing_url)

    # Make the accession number safe for file naming
    safe_accession = accession_number.replace("-", "")

    # Create a readable local file name
    file_name = f"{ticker}_{filing_date}_{safe_accession}_10K.html"

    # Create the full local file path
    file_path = SEC_RAW_DIR / file_name

    # Save the raw filing file locally
    file_path.write_text(raw_text, encoding="utf-8")

    # Track metadata about the downloaded file
    downloaded_files.append({
        "ticker": ticker,
        "company_name": company_name,
        "filing_date": filing_date,
        "accession_number": accession_number,
        "filing_url": filing_url,
        "local_file": str(file_path),
        "character_count": len(raw_text)
    })

# Convert download log into a DataFrame
downloaded_files_df = pd.DataFrame(downloaded_files)

# Display download results
downloaded_files_df

100%|██████████| 5/5 [00:01<00:00,  2.73it/s]


,ticker,company_name,filing_date,accession_number,filing_url,local_file,character_count
0,AAPL,Apple,2025-10-31,0000320193-25-000079,https://www.sec.gov/Archives/edgar/data/320193...,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,1520208
1,AMD,Advanced Micro Devices,2026-02-04,0000002488-26-000018,https://www.sec.gov/Archives/edgar/data/2488/0...,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,2174592
2,MSFT,Microsoft,2025-07-30,0000950170-25-100235,https://www.sec.gov/Archives/edgar/data/789019...,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,8158067
3,NVDA,NVIDIA,2026-02-25,0001045810-26-000021,https://www.sec.gov/Archives/edgar/data/104581...,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,1967816
4,TSLA,Tesla,2026-01-29,0001628280-26-003952,https://www.sec.gov/Archives/edgar/data/131860...,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,2391529


In [29]:
# Set output path for the download log
download_log_file = SEC_METADATA_DIR / "starter_company_10k_download_log.csv"

# Save download log to CSV
downloaded_files_df.to_csv(download_log_file, index=False)

# Confirm where the file was saved
print("Saved download log to:", download_log_file)

Saved download log to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\raw\sec_metadata\starter_company_10k_download_log.csv


In [30]:
# Select the first downloaded file
sample_file = Path(downloaded_files_df.iloc[0]["local_file"])

# Read the raw filing HTML from disk
raw_html = sample_file.read_text(encoding="utf-8")

# Print basic file information
print("File:", sample_file.name)
print("Character count:", len(raw_html))

# Preview the first 1,000 characters
print(raw_html[:1000])

File: AAPL_2025-10-31_000032019325000079_10K.html
Character count: 1520208
<?xml version='1.0' encoding='ASCII'?>
<!--XBRL Document Created with the Workiva Platform-->
<!--Copyright 2025 Workiva-->
<!--r:b93d322a-f6d3-4356-8a13-e3e1c42e12bd,g:44705cc3-5a3a-440a-975b-ed1d3694d858,d:719388195b384d85a4e238ad88eba90a-->
<html xmlns="http://www.w3.org/1999/xhtml" xmlns:dei="http://xbrl.sec.gov/dei/2025" xmlns:link="http://www.xbrl.org/2003/linkbase" xmlns:country="http://xbrl.sec.gov/country/2025" xmlns:ixt="http://www.xbrl.org/inlineXBRL/transformation/2020-02-12" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:aapl="http://www.apple.com/20250927" xmlns:xbrli="http://www.xbrl.org/2003/instance" xmlns:xbrldi="http://xbrl.org/2006/xbrldi" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:ix="http://www.xbrl.org/2013/inlineXBRL" xmlns:cyd="http://xbrl.sec.gov/cyd/2025" xmlns:srt="http://fasb.org/srt/2025" xmlns:iso4217="http://www.xbrl.org/2003/iso4217" xmlns:ixt-sec="http://www.s

In [31]:
# Parse the raw HTML with BeautifulSoup
soup = BeautifulSoup(raw_html, "lxml")

# Extract readable text from the HTML
clean_text = soup.get_text(separator=" ", strip=True)

# Print clean text length
print("Clean text character count:", len(clean_text))

# Preview the first 1,500 characters of clean text
print(clean_text[:1500])

C:\Users\tevin\AppData\Local\Temp\ipykernel_9476\1475128935.py:2: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(raw_html, "lxml")


Clean text character count: 220666
aapl-20250927 false 2025 FY 0000320193 P1Y P1Y P1Y P1Y http://fasb.org/us-gaap/2025#LongTermDebtNoncurrent http://fasb.org/us-gaap/2025#LongTermDebtNoncurrent http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent http://fasb.org/us-gaap/2025#PropertyPlantAndEquipmentNet http://fasb.org/us-gaap/2025#PropertyPlantAndEquipmentNet http://fasb.org/us-gaap/2025#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent iso4217:USD xbrli:shares iso4217:USD xbrli:shares xbrli:pure aapl:Customer aapl:Vendor aapl:Subsidiary 0000320193 2024-09-29 2025-09-27 0000320193 us-ga

In [32]:
# Create a checkpoint table showing the files and folders created
checkpoint = pd.DataFrame({
    "output": [
        "All filing metadata",
        "Latest 10-K metadata",
        "Raw 10-K filings folder",
        "Download log"
    ],
    "path": [
        str(all_filings_file),
        str(latest_10k_file),
        str(SEC_RAW_DIR),
        str(download_log_file)
    ],
    "exists": [
        all_filings_file.exists(),
        latest_10k_file.exists(),
        SEC_RAW_DIR.exists(),
        download_log_file.exists()
    ]
})

# Display the checkpoint table
checkpoint

,output,path,exists
0,All filing metadata,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True
1,Latest 10-K metadata,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True
2,Raw 10-K filings folder,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True
3,Download log,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...,True


In [33]:
# Create a summary table of what we downloaded
download_summary = downloaded_files_df[
    [
        "ticker",
        "company_name",
        "filing_date",
        "character_count",
        "local_file"
    ]
].copy()

# Sort by ticker for readability
download_summary = download_summary.sort_values("ticker").reset_index(drop=True)

# Display final download summary
download_summary

,ticker,company_name,filing_date,character_count,local_file
0,AAPL,Apple,2025-10-31,1520208,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...
1,AMD,Advanced Micro Devices,2026-02-04,2174592,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...
2,MSFT,Microsoft,2025-07-30,8158067,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...
3,NVDA,NVIDIA,2026-02-25,1967816,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...
4,TSLA,Tesla,2026-01-29,2391529,c:\Users\tevin\OneDrive\Desktop\LLM-And-Genera...
